# Coffee Maker Reviews - Text Classification with BlazingText

## About This Lab

In this notebook, we will build a **text classification** model using Amazon SageMaker's **BlazingText** algorithm. The notebook performs **binary classification** (positive vs. negative sentiment) on coffee maker reviews.

### What is BlazingText?

BlazingText is a built-in SageMaker algorithm that provides highly optimized implementations of the Word2Vec and text classification algorithms. In **supervised mode**, BlazingText works like a text classifier: it learns to associate words and phrases in a document with a label (category). Under the hood, it extends Facebook's fastText algorithm with GPU acceleration.

### What is Text Classification?

Text classification is the task of assigning a predefined category (label) to a piece of text. Common business applications include:

- **Sentiment analysis**: Is a customer review positive or negative?
- **Spam detection**: Is an email spam or legitimate?
- **Support ticket routing**: Which department should handle this customer inquiry?
- **Content moderation**: Does this social media post violate community guidelines?

### What You Will Learn

- How to split and upload BlazingText-formatted data to S3
- How to train a supervised text classification model on SageMaker
- How to deploy the model as a real-time endpoint
- How to evaluate classification performance with metrics like accuracy, precision, recall, and F1 score
- How to query the endpoint interactively with your own reviews

### Dataset

We use the **Coffee Maker Reviews** dataset, which has already been preprocessed and converted to BlazingText format in the DATA lab (NLP - Data Preparation). The preprocessing steps (lowercasing, stopword removal, lemmatization, outlier removal) and format conversion have already been applied.

- **`coffee_maker_binary_blazingtext.txt`** (binary classification): `positive`/`negative` labels

# STEP 1: IMPORT LIBRARIES AND SETUP

In [ ]:
# Install any packages not included in the SageMaker environment
!pip install seaborn -q

In [ ]:
# Install any missing dependencies that are not included in the default SageMaker environment.
import subprocess, importlib
for package in ["seaborn"]:
    if importlib.util.find_spec(package) is None:
        print(f"Installing {package}...")
        subprocess.check_call(["pip", "install", "-q", package])

# Standard Libraries
import re
import time
import logging
from collections import Counter

# Data Handling & Visualization Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn Libraries for Model Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

# NLTK Libraries for Text Preprocessing (used in STEP 12 for interactive queries)
import nltk
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

# AWS & SageMaker Libraries for Model Training and Deployment
import boto3
import sagemaker
from sagemaker import Session, get_execution_role
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Additional Libraries
from botocore.exceptions import ClientError

# Logger Setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Global Variables
file_path = "coffee_maker_binary_blazingtext.txt"
endpoint_name = "blazingtext-binaryclassification"                   # INSTRUCTION: Choose a unique endpoint name

print("All libraries imported successfully!")

# STEP 2: LOAD THE BLAZINGTEXT DATA

The text data has already been preprocessed and converted to BlazingText format in the **DATA lab** (NLP - Data Preparation notebook). Each line in the file follows this format:

```
__label__<label> cleaned text tokens here
```

**Before running this cell**, make sure you have copied your BlazingText file from the DATA lab output into the same directory as this notebook:
- **Binary**: `coffee_maker_binary_blazingtext.txt`

In [ ]:
# Read the BlazingText formatted file.
try:
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
    print("Make sure you have copied your BlazingText file from the DATA lab output.")
    raise

# Parse each line into label and text.
labels = []
texts = []
for line in lines:
    parts = line.split(" ", 1)
    labels.append(parts[0].replace("__label__", ""))
    texts.append(parts[1] if len(parts) > 1 else "")

print(f"Data loaded successfully!")
print(f"Total samples: {len(lines):,}")
print(f"Unique labels: {sorted(set(labels))}")

# Show first few lines for a quick sanity check.
print(f"\nFirst 3 lines:")
for line in lines[:3]:
    preview = line[:120] + "..." if len(line) > 120 else line
    print(f"  {preview}")

# STEP 3: VERIFY THE DATA

A quick check to confirm the BlazingText file loaded correctly. You already explored and validated this dataset in detail during the DATA lab, so here we just verify the sample count, class balance, and a few sample texts.

In [ ]:
# Quick data verification (full exploration was done in the DATA lab).
print(f"Total samples: {len(lines):,}\n")

# Class distribution.
print("Class Distribution:")
label_counts = Counter(labels)
for label in sorted(label_counts.keys()):
    count = label_counts[label]
    print(f"  {label}: {count} ({count / len(labels):.1%})")

# Sample texts per class.
unique_labels = sorted(set(labels))
for label in unique_labels:
    print(f"\n--- Sample Reviews ({label}) ---")
    shown = 0
    for lbl, txt in zip(labels, texts):
        if lbl == label and shown < 2:
            preview = txt[:120] + "..." if len(txt) > 120 else txt
            print(f"  {preview}")
            shown += 1

# STEP 4: SPLIT THE DATA

We split the data into two sets:
- **Training set (80%)**: Used to train the model.
- **Validation set (20%)**: Held out and used to monitor performance during training and for final evaluation after training.

We use **stratified splitting** to ensure that the proportion of each label is the same across both sets. This is important because an imbalanced split could bias the model.

In [ ]:
# 80/20 stratified split on the BlazingText lines.
train_lines, val_lines = train_test_split(
    lines, test_size=0.20, random_state=42, stratify=labels
)

# Write split files.
with open("train.txt", "w", encoding="utf-8") as f:
    for line in train_lines:
        f.write(line + "\n")

with open("validation.txt", "w", encoding="utf-8") as f:
    for line in val_lines:
        f.write(line + "\n")

# Parse validation lines for evaluation in STEP 8.
val_texts = []
val_labels = []
for line in val_lines:
    parts = line.split(" ", 1)
    val_labels.append(parts[0].replace("__label__", ""))
    val_texts.append(parts[1] if len(parts) > 1 else "")

# Print split statistics.
unique_labels = sorted(set(labels))
train_counts = Counter(line.split(" ", 1)[0].replace("__label__", "") for line in train_lines)
val_counts = Counter(val_labels)

print(f"Training set:   {len(train_lines)} samples")
for cls in unique_labels:
    print(f"  {cls}: {train_counts[cls]}")

print(f"\nValidation set: {len(val_lines)} samples")
for cls in unique_labels:
    print(f"  {cls}: {val_counts[cls]}")

print(f"\nFiles written: train.txt, validation.txt")

# STEP 5: UPLOAD DATA TO S3

The training and validation files are uploaded to Amazon S3, where SageMaker can access them during training.

In [ ]:
# Initialize a SageMaker session.
sagemaker_session = sagemaker.Session()

# Get the IAM role associated with SageMaker.
role = get_execution_role()

# Define the S3 bucket and region for storing data.
bucket = sagemaker_session.default_bucket()  # Default bucket for SageMaker.
region = sagemaker_session.boto_region_name   # AWS region.
prefix = "blazingtext-coffee-maker"           # S3 prefix for organizing files.

# Upload the split files to S3.
print("Uploading files to S3...")
train_uri = sagemaker_session.upload_data(path="train.txt", bucket=bucket, key_prefix=prefix)
validation_uri = sagemaker_session.upload_data(path="validation.txt", bucket=bucket, key_prefix=prefix)

# Print the S3 paths of uploaded files for reference.
print(f"\nTraining data uploaded to:   {train_uri}")
print(f"Validation data uploaded to: {validation_uri}")

# STEP 6: TRAIN THE BLAZINGTEXT MODEL

Now we train a BlazingText model in **supervised** mode using only the minimum required hyperparameter.

- **mode**: `supervised` tells BlazingText to train a text classifier. This is the only required hyperparameter. All other settings use their defaults.

| Hyperparameter | Default Value |
|---|---|
| epochs | 5 |
| learning_rate | 0.05 |
| word_ngrams | 2 |
| min_count | 5 |
| vector_dim | 100 |

In [ ]:
# Retrieve the container image URI for the BlazingText algorithm.
container = sagemaker.image_uris.retrieve(
    framework="blazingtext",
    region=region
)

# Instantiate the SageMaker Estimator for BlazingText.
blazingtext = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=sagemaker_session
)

# Set hyperparameters for the BlazingText model (minimum required).
blazingtext.set_hyperparameters(
    mode="supervised"        # Supervised text classification mode (only required hyperparameter).
)

# Create TrainingInput objects for both training and validation datasets.
train_input = TrainingInput(s3_data=train_uri, content_type="text/plain")
validation_input = TrainingInput(s3_data=validation_uri, content_type="text/plain")

# Launch the model training process.
blazingtext.fit({"train": train_input, "validation": validation_input})

print("Model training complete!")

# STEP 7: DEPLOY THE MODEL

Deploying the model creates a **real-time endpoint**, a live web service hosted on AWS that accepts text input and returns predictions. This endpoint is running on a dedicated EC2 instance, which means it incurs charges as long as it is active.

Before deploying, we check if an endpoint with the same name already exists and delete it to avoid conflicts.

In [ ]:
session = Session()                       # High-level SageMaker session.
sm_client = boto3.client("sagemaker")     # Low-level SageMaker client for endpoint management.

# Built-in waiter for polling endpoint status.
endpoint_in_service_waiter = sm_client.get_waiter("endpoint_in_service")


def delete_endpoint_and_config(endpoint_name: str, wait_for_deletion: bool = True) -> None:
    """
    Delete a SageMaker endpoint and its corresponding endpoint configuration if they exist.
    Optionally polls until resources are fully deleted.

    This function first checks if the endpoint exists. If it is in a transitional
    state (Creating or Updating), it waits for the endpoint to become InService
    before deleting. Then it deletes the endpoint configuration with the same name.

    Parameters:
        endpoint_name (str): The name of the endpoint to delete.
        wait_for_deletion (bool): Whether to poll until deletion is confirmed. Defaults to True.
    """
    # 1. Delete endpoint (if it exists).
    try:
        endpoint_desc = sm_client.describe_endpoint(EndpointName=endpoint_name)
        endpoint_status = endpoint_desc["EndpointStatus"]

        # If the endpoint is Creating or Updating, wait for it to become InService before deleting.
        if endpoint_status in ("Creating", "Updating"):
            logger.info(f"Endpoint '{endpoint_name}' is in '{endpoint_status}' state. Waiting before delete.")
            endpoint_in_service_waiter.wait(EndpointName=endpoint_name)

        # Now delete the endpoint.
        logger.info(f"Deleting endpoint: {endpoint_name}")
        sm_client.delete_endpoint(EndpointName=endpoint_name)

    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Endpoint '{endpoint_name}' does not exist or has already been deleted.")
        else:
            raise e

    # 2. Delete endpoint config (if it exists).
    try:
        sm_client.describe_endpoint_config(EndpointConfigName=endpoint_name)
        logger.info(f"Deleting endpoint configuration: {endpoint_name}")
        sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    except ClientError as e:
        if e.response["Error"]["Code"] == "ValidationException" and "Could not find" in e.response["Error"]["Message"]:
            logger.info(f"Endpoint config '{endpoint_name}' does not exist or has already been deleted.")
        else:
            raise e

    # 3. Optionally poll for deletion.
    if wait_for_deletion:
        logger.info("Waiting for endpoint & configuration to be deleted...")
        for _ in range(30):
            endpoint_exists = True
            endpoint_config_exists = True

            try:
                sm_client.describe_endpoint(EndpointName=endpoint_name)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    endpoint_exists = False

            try:
                sm_client.describe_endpoint_config(EndpointConfigName=endpoint_name)
            except ClientError as e:
                if "Could not find" in e.response["Error"]["Message"]:
                    endpoint_config_exists = False

            if not endpoint_exists and not endpoint_config_exists:
                logger.info("Endpoint and endpoint config fully deleted.")
                break

            logger.info("Endpoint or endpoint config still deleting... sleeping 10s.")
            time.sleep(10)
        else:
            logger.warning("Endpoint or endpoint config not fully deleted after 30 checks.")


# Delete any existing endpoint with the same name before deploying.
delete_endpoint_and_config(endpoint_name)

# Deploy the trained model as a real-time endpoint.
predictor = blazingtext.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name
)

# Set the serializer and deserializer for JSON input/output.
predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

logger.info(f"Endpoint '{endpoint_name}' deployed successfully and ready for inference.")

# STEP 8: EVALUATE THE DEPLOYED MODEL

Now we evaluate the model on the **validation set**. We send each validation review to the endpoint, collect the predictions, and compare them against the true labels.

We will compute the following classification metrics:

- **Accuracy**: What percentage of predictions were correct overall?
- **Precision**: Of the reviews the model predicted as positive, how many were actually positive?
- **Recall**: Of all the actually positive reviews, how many did the model correctly identify?
- **F1 Score**: The harmonic mean of precision and recall, a balanced measure of model quality.

In [ ]:
def evaluate_model(predictor, eval_texts, eval_labels, batch_size=100):
    """
    Evaluate the deployed BlazingText model on a set of texts and labels.

    This function sends texts to the deployed endpoint in batches, collects
    the predicted labels and confidence scores, and computes classification metrics.
    It handles both binary and multiclass classification dynamically.

    Parameters:
        predictor: The SageMaker Predictor object for the deployed endpoint.
        eval_texts (list): The review texts to evaluate.
        eval_labels (list): The true labels for the reviews.
        batch_size (int, optional): Number of reviews to send per prediction request. Defaults to 100.

    Returns:
        dict: A dictionary containing:
            - "metrics": dict with Accuracy, Precision, Recall, F1 Score
            - "predicted_labels": list of predicted label strings
            - "confidences": list of confidence scores (floats)
            - "true_labels": list of true label strings
            - "texts": list of review text strings
    """
    texts = list(eval_texts)
    true_labels = [str(label) for label in eval_labels]
    predicted_labels = []
    confidences = []

    # Send predictions in batches to avoid payload size limits.
    for start_i in range(0, len(texts), batch_size):
        end_i = min(start_i + batch_size, len(texts))
        batch_texts = texts[start_i:end_i]

        # BlazingText expects a JSON payload with an "instances" key.
        payload = {"instances": batch_texts}
        response = predictor.predict(payload)

        # Extract the predicted label and confidence from each prediction.
        for prediction in response:
            label_str = prediction["label"][0]
            predicted_label = label_str.replace("__label__", "")
            predicted_labels.append(predicted_label)
            confidences.append(float(prediction["prob"][0]))

    # Determine the unique class labels for dynamic metric computation.
    all_labels = sorted(set(true_labels) | set(predicted_labels), key=str)
    num_classes = len(all_labels)
    is_binary = (num_classes == 2)

    # Compute classification metrics.
    acc = accuracy_score(true_labels, predicted_labels)

    if is_binary:
        pos_label = all_labels[1]
        prec = precision_score(true_labels, predicted_labels, pos_label=pos_label, zero_division=0)
        rec = recall_score(true_labels, predicted_labels, pos_label=pos_label, zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, pos_label=pos_label, zero_division=0)
    else:
        prec = precision_score(true_labels, predicted_labels, average="weighted", zero_division=0)
        rec = recall_score(true_labels, predicted_labels, average="weighted", zero_division=0)
        f1 = f1_score(true_labels, predicted_labels, average="weighted", zero_division=0)

    # Print the evaluation metrics.
    print("\nEvaluation Metrics on Validation Set:")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")

    # Print plain-language interpretation of each metric.
    print("\nInterpretation of Metrics:")
    print(f"  - Accuracy ({acc:.1%}): Out of all predictions, {acc:.1%} were correct.")
    print(f"  - Precision ({prec:.1%}): When the model predicted a class, it was right {prec:.1%} of the time.")
    print(f"  - Recall ({rec:.1%}): The model correctly identified {rec:.1%} of actual instances per class.")
    print(f"  - F1 Score ({f1:.4f}): A balanced measure combining precision and recall (1.0 is perfect).")

    # Print the full classification report.
    print("\nClassification Report:")
    print(classification_report(true_labels, predicted_labels, target_names=all_labels))

    # Create and display the confusion matrix as a heatmap.
    cm = confusion_matrix(true_labels, predicted_labels, labels=all_labels)
    cm_percent = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_percent, annot=True, fmt=".2%", cmap="Blues",
                xticklabels=all_labels,
                yticklabels=all_labels)
    plt.title("Confusion Matrix (Percentage)")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.show()

    return {
        "metrics": {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1 Score": f1},
        "predicted_labels": predicted_labels,
        "confidences": confidences,
        "true_labels": true_labels,
        "texts": texts
    }


# Evaluate the deployed model on the validation set.
results = evaluate_model(predictor, val_texts, val_labels)
print(results["metrics"])

# STEP 9: ANALYZE MISCLASSIFIED EXAMPLES

No model is perfect. Understanding **where** the model gets it wrong is just as important as knowing its overall accuracy. By looking at misclassified reviews, we can spot patterns in the types of mistakes the model makes.

Common reasons models misclassify text:
- **Sarcasm or irony**: "Oh great, another coffee maker that leaks" is negative but uses a positive word ("great").
- **Mixed sentiment**: "The coffee tastes amazing but the machine broke after a week" contains both positive and negative language.
- **Ambiguous wording**: Short or vague reviews may not contain enough signal for the model to decide confidently.

In [ ]:
# Build a DataFrame of all predictions for analysis.
pred_df = pd.DataFrame({
    "text": results["texts"],
    "true_label": results["true_labels"],
    "predicted_label": results["predicted_labels"],
    "confidence": results["confidences"]
})

# Flag correct vs incorrect predictions.
pred_df["correct"] = pred_df["true_label"] == pred_df["predicted_label"]

# Filter to misclassified examples only.
misclassified = pred_df[~pred_df["correct"]].copy()
print(f"Total validation reviews: {len(pred_df)}")
print(f"Misclassified: {len(misclassified)} ({len(misclassified) / len(pred_df):.1%})\n")

# Group misclassifications by error type.
if len(misclassified) > 0:
    error_groups = misclassified.groupby(["true_label", "predicted_label"]).size().reset_index(name="count")
    error_groups = error_groups.sort_values("count", ascending=False)

    for _, row in error_groups.iterrows():
        print(f"Error type: Predicted '{row['predicted_label']}', actually '{row['true_label']}' ({row['count']} cases)")
        subset = misclassified[
            (misclassified["true_label"] == row["true_label"]) &
            (misclassified["predicted_label"] == row["predicted_label"])
        ].head(5)
        for _, ex in subset.iterrows():
            preview = ex["text"][:100] + "..." if len(str(ex["text"])) > 100 else ex["text"]
            print(f"  [{ex['confidence']:.0%} confidence] {preview}")
        print()
else:
    print("No misclassified examples found. The model classified every validation review correctly.")

# STEP 10: PREDICTION CONFIDENCE ANALYSIS

The model does not just predict a label; it also reports a **confidence score** (probability) for each prediction. High confidence means the model is very sure about its answer; low confidence means it is uncertain.

Comparing confidence scores for correct vs. incorrect predictions tells us something important: if the model tends to be less confident when it is wrong, that is a good sign. It means we could potentially set a **confidence threshold** in a business application (for example, only auto-classify reviews where the model is more than 90% confident, and send the rest to a human reviewer).

In [ ]:
# Split confidence scores by correct vs incorrect predictions.
correct_conf = pred_df[pred_df["correct"]]["confidence"]
incorrect_conf = pred_df[~pred_df["correct"]]["confidence"]

# Histogram of confidence scores.
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(correct_conf, bins=20, alpha=0.7, label=f"Correct ({len(correct_conf)})", color="green", edgecolor="white")
if len(incorrect_conf) > 0:
    ax.hist(incorrect_conf, bins=20, alpha=0.7, label=f"Incorrect ({len(incorrect_conf)})", color="red", edgecolor="white")
ax.set_title("Confidence Score Distribution: Correct vs. Incorrect Predictions")
ax.set_xlabel("Confidence Score")
ax.set_ylabel("Number of Predictions")
ax.legend()
plt.tight_layout()
plt.show()

# Summary statistics.
print("Confidence Summary Statistics:")
print(f"  Correct predictions   - Mean: {correct_conf.mean():.4f}, Median: {correct_conf.median():.4f}")
if len(incorrect_conf) > 0:
    print(f"  Incorrect predictions - Mean: {incorrect_conf.mean():.4f}, Median: {incorrect_conf.median():.4f}")
    gap = correct_conf.mean() - incorrect_conf.mean()
    print(f"\n  Confidence gap (correct - incorrect): {gap:.4f}")
    if gap > 0.05:
        print("  The model is noticeably more confident when it is correct. This is a good sign.")
    else:
        print("  The model shows similar confidence for both correct and incorrect predictions.")
else:
    print("  No incorrect predictions to compare against.")

# STEP 11: BASELINE COMPARISON

Before celebrating a model's accuracy, always ask: **how much better is it than simply guessing?**

The simplest baseline is the **majority-class classifier**, which always predicts the most common label. For example, if 70% of reviews are positive, a model that always predicts "positive" would achieve 70% accuracy without learning anything at all.

Comparing the model against this baseline shows the **lift**: how much value the model adds beyond naive guessing. A model that barely beats the baseline may not be worth the cost of deploying and maintaining it.

In [ ]:
# Compute the majority-class baseline accuracy.
true_labels = results["true_labels"]
label_counts = pd.Series(true_labels).value_counts()
majority_label = label_counts.index[0]
baseline_accuracy = label_counts.iloc[0] / len(true_labels)
model_accuracy = results["metrics"]["Accuracy"]

print(f"Majority class: '{majority_label}' ({label_counts.iloc[0]} of {len(true_labels)} validation reviews)")
print(f"Baseline accuracy (always predict '{majority_label}'): {baseline_accuracy:.1%}")
print(f"Model accuracy: {model_accuracy:.1%}")

# Calculate lift.
lift = model_accuracy - baseline_accuracy
lift_pct = (lift / baseline_accuracy) * 100 if baseline_accuracy > 0 else 0
print(f"\nLift over baseline: +{lift:.1%} (a {lift_pct:.1f}% relative improvement)")

# Side-by-side bar chart.
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    ["Majority-Class Baseline", "BlazingText Model"],
    [baseline_accuracy, model_accuracy],
    color=["#d3d3d3", "#4a90d9"],
    edgecolor="white",
    width=0.5
)

# Add value labels on the bars.
for bar, val in zip(bars, [baseline_accuracy, model_accuracy]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.1%}", ha="center", fontweight="bold", fontsize=13)

ax.set_ylabel("Accuracy")
ax.set_title("Model Accuracy vs. Majority-Class Baseline")
ax.set_ylim(0, 1.1)
ax.axhline(y=baseline_accuracy, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Business interpretation.
if lift > 0.05:
    print("The model provides meaningful lift over the baseline. It is learning useful patterns from the text.")
elif lift > 0:
    print("The model beats the baseline, but the improvement is modest. Consider additional tuning or more training data.")
else:
    print("The model does not beat the baseline. It may need more data, different features, or a different approach.")

# STEP 12: QUERY THE ENDPOINT WITH CUSTOM TEXT

Now you can test the model interactively! Type your own coffee maker review and see how the model classifies it. The model will return:
- The **predicted label** (e.g., positive/negative, or a star rating)
- The **confidence score** (how certain the model is about its prediction)

**Important**: Your input is automatically preprocessed using the same pipeline from the DATA lab (lowercasing, stopword removal, lemmatization, etc.) before being sent to the model. This ensures the model sees text in the same format it was trained on.

Type `quit` to exit the loop.

In [ ]:
# Text preprocessing tools (same pipeline used in the DATA lab).
word_tokenizer = RegexpTokenizer(r"\w+")
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def _get_wordnet_pos(treebank_tag):
    """Map a Penn Treebank POS tag to the corresponding WordNet POS constant."""
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def preprocess_text(text):
    """
    Apply the same preprocessing pipeline used in the DATA lab so that
    interactive queries match the format the model was trained on.

    Steps: lowercase, remove URLs/HTML, tokenize, remove single-character
    tokens, POS-tag, remove stopwords, lemmatize, rejoin.
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    tokens = word_tokenizer.tokenize(text)
    tokens = [t for t in tokens if len(t) > 1]
    tagged = nltk.pos_tag(tokens)
    tagged = [(w, pos) for w, pos in tagged if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w, _get_wordnet_pos(pos)) for w, pos in tagged]
    return " ".join(tokens)


print("Enter a coffee maker review to classify (type 'quit' to exit):")
print("=" * 60)

while True:
    # Prompt the user for a review.
    user_input = input("\nYour review: ").strip()

    # Exit the loop if the user types 'quit'.
    if user_input.lower() == "quit":
        print("Exiting prediction loop.")
        break

    # Skip empty inputs.
    if not user_input:
        print("Please enter a review.")
        continue

    # Preprocess the input using the same pipeline the training data went through.
    cleaned_input = preprocess_text(user_input)
    print(f"  Preprocessed: {cleaned_input}")

    # Send the preprocessed review to the endpoint.
    payload = {"instances": [cleaned_input]}
    response = predictor.predict(payload)

    # Extract the prediction.
    prediction = response[0]
    label_str = prediction["label"][0]
    confidence = float(prediction["prob"][0])
    predicted_label = label_str.replace("__label__", "")

    # Display the result.
    print(f"  Prediction: {predicted_label.upper()} (confidence: {confidence:.2%})")

# STEP 13: DELETE THE ENDPOINT (CLEANUP)

**WARNING: The deployed endpoint is running on a live EC2 instance and incurs charges as long as it is active. Always delete the endpoint when you are done to avoid unexpected AWS costs.**

This step asks for confirmation before deleting. If you choose not to delete now, remember to come back and delete the endpoint manually when you are finished experimenting.

In [ ]:
# Interactive confirmation before deleting the endpoint.
confirm = input("Do you want to delete the endpoint? (yes/no): ").strip().lower()

if confirm == "yes":
    delete_endpoint_and_config(endpoint_name)
    print(f"\nEndpoint '{endpoint_name}' has been deleted.")
else:
    print(f"\nEndpoint NOT deleted.")
    print("Remember to delete it manually when you are done to avoid ongoing AWS charges.")
    print(f"You can re-run this cell or call: delete_endpoint_and_config('{endpoint_name}')")